# Chapter 4: Storage and Retrieval
*From: Designing Data-Intensive Applications*

---

## TL;DR

- A storage engine has one job — **put bytes in, get them back out quickly** — but the right internal structure depends entirely on whether the workload is OLTP (many small point reads and writes on the current state) or OLAP (few large scans over history).
- On the OLTP side, two families dominate: **log-structured (LSM-trees)** that append sorted segments and compact them in the background, and **update-in-place B-trees** that overwrite fixed-size pages and use a write-ahead log for crash safety.
- As a rule of thumb, **LSM wins writes** (sequential I/O, lower write amplification), **B-trees win reads** (predictable latency, no segment fan-out), but the gap narrows on NVMe and with careful tuning.
- On the OLAP side, **column-oriented storage** flips the layout 90° — values of a single column are stored together, compressed with bitmap and run-length encodings, and scanned with **vectorized or JIT-compiled operators** for SIMD-friendly tight loops.
- Beyond 1-D keys, specialized structures answer multidimensional queries: **R-trees** for geospatial, **inverted indexes** (postings lists / Lucene) for full-text search, and **IVF / HNSW** for approximate nearest-neighbor search over vector embeddings.

---

## Problem Framing

A storage engine has a deceptively simple contract: accept a write and return the right value on a later read. Chapter 4 is about *how* that contract is kept on hardware that is nothing like RAM — disks that reward sequential over random I/O, pages that can tear during a crash, SSDs whose flash blocks wear out, and object storage whose latency is measured in tens of milliseconds. Every storage engine trades **read throughput, write throughput, space amplification, and predictability** against one another; the chapter catalogs these trade-offs.

The single most important framing is the **OLTP/OLAP split**. They are not two operating points of the same engine; they are different machines:

| Property | OLTP (operational) | OLAP (analytical) |
|---|---|---|
| Read pattern | Point / short-range lookup by key | Aggregates scanning millions–billions of rows |
| Write pattern | Many small create / update / delete | Bulk loads (ETL) or streaming ingest |
| Hot dataset | Gigabytes → a few terabytes | Terabytes → petabytes |
| Rows accessed / query | ~1 | Millions+ |
| Columns accessed / query | All of the row | 4–5 out of 100+ |
| Latency target | ms | seconds → minutes |
| Storage layout | **Row-oriented**, indexed by key | **Column-oriented**, compressed, sorted |

Holding onto this split while reading the rest of the chapter is the key — LSM-trees, B-trees, and their indexes are all OLTP machinery; column stores, vectorized execution, and data cubes are all OLAP machinery; multidimensional, full-text, and vector indexes are the specialist shapes that neither family handles well by default.

---

## Map of the Chapter

```mermaid
graph TB
  ROOT["Chapter 4: Storage and Retrieval"]
  ROOT --> OLTP["OLTP Storage Engines"]
  ROOT --> OLAP["OLAP Storage Engines"]
  ROOT --> SPEC["Specialist Indexes"]

  OLTP --> LSM["1. Log-Structured Storage / LSM-Trees<br/>memtable + SSTables + compaction + Bloom filters"]
  OLTP --> BT["2. B-Trees and Page-Oriented Storage<br/>fixed pages, update-in-place, WAL"]
  OLTP --> CMP["3. Comparing B-Trees and LSM-Trees<br/>reads vs writes, amplification, fragmentation"]
  OLTP --> IDX["4. Secondary, Clustered, Covering Indexes<br/>heap vs clustered vs covering"]

  OLAP --> COL["5. Column-Oriented Storage<br/>columnar layout, bitmap + RLE compression, sort order"]
  OLAP --> QE["6. Query Execution &amp; Materialized Views<br/>JIT codegen vs vectorization; cubes"]

  SPEC --> MD["7. Multidimensional, Full-Text, Vector Indexes<br/>R-tree / inverted index / IVF / HNSW"]

  LSM --> CMP
  BT --> CMP
  BT --> IDX
  COL --> QE
  LSM --> MD
```

The seven concepts line up with the chapter's sections. Read top-to-bottom: the OLTP pair (LSM vs B-tree) + their comparison + how secondary indexes layer on top of either family; then the OLAP analog (columnar storage + the query engines built on it); finally the specialist indexes for queries a 1-D B-tree cannot answer.

---

## A Back-of-the-Envelope: Why B-Trees Scale, Bloom Filters Work, and LSMs Beat B-Trees on Writes

The chapter drops three memorable numbers: a four-level B-tree with branching factor 500 and 4 KiB pages holds 250 TB; 10 bits per key of Bloom filter gives a 1% false-positive rate; and LSM-trees typically have lower write amplification than B-trees because they do not rewrite whole pages. Let's make each concrete.

In [ ]:
# ---- 1. B-tree capacity: why depth 4 is enough ----
page_size_bytes = 4 * 1024          # 4 KiB page
branching_factor = 500              # children per interior page (typical)
depth = 4                           # root + 2 interior + leaf

leaf_pages = branching_factor ** (depth - 1)
leaf_bytes = leaf_pages * page_size_bytes
leaf_tb = leaf_bytes / 1e12

print("=== B-tree capacity ===")
print(f"Depth:                 {depth}")
print(f"Branching factor:      {branching_factor}")
print(f"Leaf pages:            {leaf_pages:>18,}")
print(f"Total capacity:        {leaf_tb:>18,.0f} TB")
print(f"Pages read per lookup: {depth}  (one per level, ~log_500 of the dataset)")

# ---- 2. Bloom filter sizing: the 10-bits/key rule ----
# The chapter's rule of thumb: 10 bits/key -> ~1% false positive,
# and every extra 5 bits/key cuts FP rate by ~10x.
keys_in_sstable = 10_000_000
for bits_per_key in (5, 10, 15, 20):
    bytes_used = keys_in_sstable * bits_per_key / 8
    # Rough approximation matching the chapter's rule of thumb.
    fp_rate = 10 ** (-(bits_per_key - 5) / 5)
    print(f"Bloom filter @ {bits_per_key:>2} bits/key: "
          f"{bytes_used/1e6:>7.1f} MB, FP ~{fp_rate*100:>6.2f}%")

# ---- 3. Write amplification: LSM (leveled) vs B-tree ----
# B-tree: each logical write touches a WAL record + a full page rewrite.
# LSM: each value is written to the WAL once, flushed to L0 once, then
# re-written every time it is compacted to a deeper level.
record_size = 200             # bytes of user data per write
page_size   = 4096            # B-tree page size

# B-tree: WAL entry + full-page write-back (+ occasionally a double-write buffer).
btree_bytes_per_write = record_size + page_size   # ~4.3 KB on disk per 200 B write
btree_wa = btree_bytes_per_write / record_size

# LSM with leveled compaction at size ratio 10, 5 levels deep.
# A value is rewritten once per level it traverses on its way to the bottom.
lsm_levels = 5
lsm_wa = 1 + lsm_levels        # WAL (1) + one rewrite per level

print()
print("=== Write amplification (bytes written / bytes of user data) ===")
print(f"B-tree (WAL + full page): {btree_wa:>5.1f}x")
print(f"LSM (leveled, {lsm_levels} levels):  {lsm_wa:>5.1f}x")
print(f"LSM advantage on writes:  ~{btree_wa/lsm_wa:>4.1f}x less disk bandwidth per write")

The three numbers tell the whole story of OLTP storage: a B-tree reaches hundreds of terabytes in just four page reads per lookup; a tiny Bloom filter lets an LSM skip most of its segments on a missing key; and LSMs win writes by amortizing many small writes into a few big sequential compactions, while B-trees pay a full-page overhead every time.

---

## Deep Dive 1 — Log-Structured Storage and LSM-Trees

```mermaid
graph TB
  subgraph Memory
    MT["Memtable<br/>(sorted: red-black tree / skip list)"]
  end
  W[Write] --> MT
  W --> WAL[(WAL on disk)]
  MT -- "flush when full" --> S0[SSTable L0]
  S0 --> C["Background compaction"]
  S1[SSTable L1] --> C
  S2[SSTable L2] --> C
  C --> SNEW["New merged SSTable<br/>(sorted, immutable)"]

  R[Read] --> MT
  R --> BF{"Bloom filter per SSTable"}
  BF -- miss --> SKIP["skip segment"]
  BF -- hit --> BLK["Sparse index -> block -> value<br/>(newest-to-oldest order)"]
```

The write path of an LSM engine treats disk the way sane people treat filesystems: never modify what is already there. Writes go to an in-memory ordered structure (the **memtable**) and a crash-recovery log; when the memtable fills, it is flushed to disk as an immutable **SSTable** — a file of key-value pairs sorted by key with a sparse block index. Over time, SSTables accumulate, so a background process **merges and compacts** them, keeping the newest value for each key and dropping tombstones. Reads consult the memtable and then the SSTables newest-to-oldest, using a **Bloom filter** per segment to skip files that certainly do not contain the key.

Two standard compaction strategies trade disk space for read/write balance. **Size-tiered** compaction merges same-sized SSTables into ever-larger ones — it handles very high write throughput because data gets rewritten only a few times, but produces giant files that temporarily double disk usage during compaction. **Leveled** compaction fixes SSTable size and stacks them into levels L0, L1, L2…, each ~10x the size of the previous; a level's files cover disjoint key ranges so reads hit fewer files and overall space amplification is lower, at the cost of more constant compaction churn. RocksDB, Cassandra, ScyllaDB, HBase, and Lucene's segment merging are all variants on this theme.

LSM's key insight is that **sequential writes dominate random writes on both HDDs and SSDs** — the flash translation layer's block-erase unit (~512 KiB) punishes random writes with garbage collection, while whole-segment writes let the FTL retire blocks cleanly. This is why LSM engines can happily write to object storage (SlateDB, Delta Lake), whose API is essentially "append immutable files".

---

## Deep Dive 2 — B-Trees and Page-Oriented Storage

```mermaid
graph TB
  ROOT["Root page<br/>[100, 200, 300, ...]"]
  ROOT -- ">= 200, < 300" --> I1["Interior page<br/>[210, 240, 270]"]
  I1 -- ">= 240, < 270" --> L1["Leaf page<br/>[key=251, value=...]"]

  subgraph Crash Safety
    WAL[(Write-ahead log)]
    PAGES[(B-tree pages)]
    WAL -. "replay on recovery" .-> PAGES
  end

  W[Write key=251] --> WAL
  W --> PAGES

  SPLIT["Leaf overflow -> split<br/>parent page updated to reference both halves"]
  L1 -. "if full" .-> SPLIT
```

A B-tree slices the database into **fixed-size pages** (4 KiB in classic designs, 8 KiB in Postgres, 16 KiB in InnoDB) arranged as a balanced tree. An interior page holds sorted key boundaries and child-page pointers; a leaf page holds either the actual rows (clustered) or pointers to them (heap file). Lookups walk root → interior → leaf — four page reads is enough for 250 TB when the branching factor is ~500. Inserts and updates **overwrite pages in place**; when a leaf overflows, it splits and the parent is updated, recursively up to a new root if needed.

In-place overwrite is fast but dangerous: a crash mid-split leaves the tree inconsistent, and modern hardware cannot atomically write a whole 4 KiB page (the "**torn page**" problem). The standard solution is a **write-ahead log**: every modification is appended to the WAL and fsync'd before any B-tree page is overwritten; on crash, the engine replays the WAL. LMDB takes a different route — **copy-on-write**: modified pages are written to a new location and a new root is published, giving crash safety and cheap snapshots for free, at the cost of more disk usage and a different concurrency model.

Several optimizations are near-universal. **Key abbreviation** in interior pages packs more children per page, raising the branching factor and lowering tree depth. **Sibling pointers** on leaf pages speed up range scans without climbing back up the tree. **Buffering writes in memory** and flushing pages lazily — relying on the WAL for durability — smooths the random-write cost.

---

## Deep Dive 3 — Comparing B-Trees and LSM-Trees

```mermaid
graph LR
  subgraph Reads
    BTR["B-tree<br/>+ predictable: O(log) page reads<br/>+ range scans natural<br/>- pointer-chasing"]
    LTR["LSM-tree<br/>- may check many SSTables<br/>+ Bloom filters mask cost<br/>- range scans merge many segments"]
  end

  subgraph Writes
    BTW["B-tree<br/>- random writes + full-page rewrite<br/>- WAL doubles bytes<br/>- high write amplification"]
    LTW["LSM-tree<br/>+ sequential segment writes<br/>+ lower write amplification<br/>- latency spikes during compaction"]
  end

  subgraph Disk usage
    BTD["B-tree<br/>- fragmentation over time<br/>- vacuum / repack needed"]
    LTD["LSM-tree<br/>+ compaction rewrites cleanly<br/>+ compression-friendly blocks<br/>- size-tiered doubles during merge"]
  end
```

On **reads**, B-trees have a tidy story: a handful of page fetches walks root-to-leaf and the data is there. LSM reads may have to consult the memtable, the top-level SSTable, and potentially deeper levels — Bloom filters make the common miss path nearly free, but range scans cannot use them and must merge results from all segments in parallel. For pure point-lookup workloads on hot data, both families are fast; for range-heavy or unpredictable-latency-averse workloads, B-trees tend to win.

On **writes**, LSM-trees win on throughput and SSD lifespan for two compounding reasons. First, LSMs do **sequential segment writes** — even on SSDs, which have no mechanical penalty, a sequential write fills whole flash blocks cleanly and avoids FTL garbage collection. Second, LSMs have **lower write amplification**: a B-tree must write a WAL entry and a whole 4 KiB page for every tiny update; an LSM writes the record to WAL + memtable flush + one rewrite per compaction level (~5–6x total). The catch is predictability: a large compaction can cause **backpressure stalls** where reads and writes pause; B-trees spread their cost more evenly.

On **disk space**, LSMs with leveled compaction usually use less space (better compression in contiguous blocks, no fragmentation from deleted keys) but size-tiered compaction briefly doubles usage during big merges. B-trees suffer **fragmentation** as pages half-empty after deletes — Postgres's VACUUM exists to reclaim that space. **Snapshots** are trivially cheap on LSM (segments are immutable; record the manifest) and expensive on B-trees (pages get overwritten). Finally, **deletions** are instant in a B-tree but can linger in deep LSM levels until a compaction propagates the tombstone — a real concern under data-protection regulations.

---

## Deep Dive 4 — Secondary, Clustered, and Covering Indexes

```mermaid
graph TB
  subgraph Heap file + secondary index
    H["Heap file (rows in insert order)"]
    IDX1["Secondary index<br/>key -> row pointer"]
    IDX1 --> H
  end

  subgraph Clustered index
    CI["Primary key B-tree<br/>leaf pages hold the full row"]
    SI["Secondary index<br/>key -> primary key"]
    SI --> CI
  end

  subgraph Covering index
    CV["Covering index<br/>key + frequently-read columns<br/>answers query without a second hop"]
  end
```

Real tables have more than one column, and applications rarely query only by the primary key. A **secondary index** is a separate sorted structure mapping a non-primary attribute to the rows that contain it — implementable on either a B-tree or an LSM. Because the indexed value is not unique, each index entry either points to a list of row IDs or appends the row ID to the key to keep entries unique (like postings lists in full-text search).

The choice of where the **row itself** lives drives three common patterns. A **heap file** stores rows in arrival order; every index, primary or secondary, holds a pointer into the heap. Updates that grow a row may move it, forcing all indexes to update or leaving a forwarding pointer behind (Postgres takes this path). A **clustered index** puts the actual row data in the leaves of the primary-key B-tree; secondary indexes point to the primary key, requiring a second lookup (MySQL InnoDB). A **covering index** (or index-with-included-columns) splits the difference — the index holds the search key plus the handful of extra columns the query needs, so the engine can answer a query from the index alone (SQL Server, Postgres with INCLUDE). Each extra column in an index costs disk space and slows writes, but can cut read latency dramatically.

| Aspect | Heap file + pointer indexes | Clustered index | Covering index |
|---|---|---|---|
| Primary lookup | Index → heap (two hops) | One B-tree walk | One B-tree walk |
| Secondary lookup | Index → heap (two hops) | Secondary → primary → leaf (three hops) | Index only (one hop) if query columns covered |
| Insert cost | Low | Higher (keep heap in key order via splits) | Higher per extra column |
| Update-in-place | Easy unless row grows | Rows fixed to key position | Index must be updated for any covered column |
| Space overhead | Low | Rows stored once | Extra columns duplicated |
| Example | PostgreSQL | MySQL InnoDB primary keys | SQL Server / Postgres INCLUDE |

---

## Deep Dive 5 — Column-Oriented Storage for Analytics

```mermaid
graph TB
  subgraph "Row store (OLTP)"
    R["Row 1: [date, product, store, qty, price, customer, ...]<br/>Row 2: [date, product, store, qty, price, customer, ...]<br/>Row 3: [date, product, store, qty, price, customer, ...]"]
  end

  subgraph "Column store (OLAP)"
    C1["date:    [d1, d1, d2, d2, d3, d3, d3, d4, ...]  (sorted, RLE)"]
    C2["product: [p1, p2, p1, p3, p1, p2, p3, p1, ...]  (bitmap per product)"]
    C3["qty:     [1, 3, 2, 1, 5, 1, 1, 2, ...]          (delta-encoded)"]
    DOTS["...one file / stripe per column per row-range..."]
  end

  Q["SELECT SUM(qty) WHERE date IN ... AND product IN ..."] --> C1
  Q --> C2
  Q --> C3
```

OLTP loads a whole row into cache to read a few fields; OLAP reads billions of rows but touches only a handful of columns. Column-oriented storage rotates the layout 90°: for each column, all values across all rows are stored contiguously, in the same row order so the k-th entry of every column belongs to the k-th row. Parquet, ORC, Lance, and Nimble are disk formats in this family; Arrow is the in-memory analog; engines like Snowflake, BigQuery, DuckDB, ClickHouse, Druid, and Pinot all build on it. The layout pays off because analytical queries scan only the relevant column files, skip everything else, and enjoy exceptionally compressible sequences of similar values.

Two compression techniques recur. **Bitmap encoding** represents a column with N distinct values as N bitmaps — one bit per row, 1 where the row has that value. A `WHERE product IN (31, 68, 69)` clause becomes the bitwise OR of three bitmaps; a `WHERE product=X AND store=Y` clause becomes a bitwise AND. **Run-length encoding** (often with roaring bitmaps that switch representations adaptively) collapses long runs of identical values, which are common in low-cardinality columns and extremely common after sorting.

**Sort order** is the design knob. The first sort key gets the best compression because long runs of identical values collapse dramatically; subsequent sort keys compress progressively less. Choose the first sort key to match the dominant filter (often `date_key` for time-series-shaped queries). Writes are handled by the same log-structured trick used for OLTP LSMs: accumulate row-wise writes in memory, then bulk-rewrite column files during compaction — rewriting mid-file into compressed columns is not practical, but a bulk merge is.

---

## Deep Dive 6 — Query Execution: Compilation, Vectorization, and Materialized Views

```mermaid
graph LR
  SQL[SQL query] --> PLAN["Query plan<br/>(operators, join order)"]

  PLAN --> CG["Path A: Query compilation<br/>generate C / LLVM code<br/>JIT to machine code<br/>(Hyper, Spark Tungsten)"]
  PLAN --> VEC["Path B: Vectorized execution<br/>prebuilt operators on<br/>batches of column values<br/>(MonetDB/X100, DuckDB, ClickHouse)"]

  CG --> CPU[Hot inner loop, SIMD, no branches]
  VEC --> CPU

  subgraph Pre-aggregation
    MV["Materialized view<br/>(stored query result)"]
    CUBE["OLAP cube<br/>(precomputed aggregates along<br/>date x product x store x ...)"]
  end

  CPU --> RES[Result]
  MV --> RES
  CUBE --> RES
```

Scanning columns fast is not enough; you also have to process them fast. A naive row-by-row interpreter that checks "which op next?" for every value is cache- and branch-predictor-hostile. Two schools of thought both fix this.

**Query compilation** translates the SQL query into native code — often via LLVM — producing a tight loop that does exactly the comparisons this query needs and nothing more. It removes interpreter overhead entirely but adds compile latency; good for long-running analytical queries, less good for subsecond queries. **Vectorized processing** keeps an interpreter but makes each operator work on a **batch of column values** at a time: `product_sk == 'bananas'` produces a 1024-bit bitmap, `store_sk == 3` produces another, and a bitwise AND yields the result. Each operator's inner loop is small, fits in L1, uses SIMD instructions, and operates directly on compressed representations when possible. Both approaches exploit the same modern-CPU tricks (sequential memory, tight loops, SIMD, compressed-data operations); compilation is common in Hyper/SingleStore, vectorization in MonetDB/X100, DuckDB, and ClickHouse.

For repeatedly executed queries, neither fast path beats **not executing at all**. A **materialized view** stores the result of a query as a real table and incrementally updates it when underlying data changes — great for dashboards that run the same query every minute. Its specialized cousin, the **OLAP cube** or **data cube**, pre-aggregates a measure (e.g., SUM(net_price)) across every combination of chosen dimensions; once built, a "total sales by store per day" query is a constant-time lookup along one axis of the cube. The trade-off is that cubes answer only the queries along their chosen dimensions — you cannot ask "what fraction of sales came from items over $100" if `price_band` is not a dimension — and every write must update every containing aggregate.

---

## Deep Dive 7 — Multidimensional, Full-Text, and Vector Indexes

```mermaid
graph TB
  subgraph Multidimensional
    RT["R-tree / Bkd-tree<br/>partitions 2-D space into<br/>nested bounding boxes"]
    RT --> GEO["lat/lon range query:<br/>all restaurants in this map rectangle"]
  end

  subgraph Full-text
    INV["Inverted index<br/>term -> postings list (bitmap of doc IDs)"]
    INV --> AND["AND / OR of term bitmaps<br/>(+ trigrams for substring,<br/>Levenshtein automaton for typos)"]
  end

  subgraph Vector
    EMB["Embedding model (LLM / BERT / CLIP)"]
    EMB --> VEC["Vector embedding (1000+ floats)"]
    VEC --> IDX{Vector index}
    IDX --> FLAT["Flat: exact, O(N)"]
    IDX --> IVF["IVF: partition into centroids,<br/>query 'probes' k partitions"]
    IDX --> HNSW["HNSW: multi-layer proximity graph,<br/>coarse top layer -> dense bottom"]
  end
```

A 1-D B-tree can find all last names starting with "L" or all timestamps in a day, but it cannot find all points inside a map rectangle or all documents containing two keywords. Three very different structures fill that gap.

**Multidimensional** indexes handle queries that constrain multiple attributes at once. A concatenated index on `(latitude, longitude)` cannot do a true 2-D range query — it can narrow one axis but not both simultaneously. **R-trees** and **Bkd-trees** recursively partition space into nested bounding boxes (or KD-tree splits) so nearby points live in the same subtree; PostGIS uses an R-tree via Postgres's GiST framework. The same structures work for any low-dimensional range query: `(date, temperature)`, `(red, green, blue)` for color search, and so on. An alternative is to collapse 2-D into 1-D with a **space-filling curve** (Z-order, Hilbert) and use a regular B-tree.

**Full-text** indexes treat each possible term as a dimension and each document as a vector of 0s and 1s. The workhorse structure is the **inverted index** — a sorted map from term to postings list, where the postings list is either a list of document IDs or a compressed bitmap over document IDs. "red AND apples" becomes a bitwise AND between two postings bitmaps, exactly the vectorized pattern from the OLAP section. Lucene (used by Elasticsearch and Solr) stores these postings lists in SSTable-like files merged log-structured-style; Postgres's GIN index does the same inside JSON. Trigram indexes extend this to arbitrary substring and regex search; Levenshtein automata extend it to fuzzy matching.

**Vector** indexes answer semantic similarity queries by embedding documents and queries into a high-dimensional (~1000+) floating-point space and returning nearest neighbors. R-trees collapse above ~10 dimensions, so purpose-built structures take over. A **flat** index stores all vectors and does exact O(N) scans — fine below a few hundred thousand vectors. **IVF** (inverted file) clusters the vector space into centroids and only compares the query to vectors in the nearest `probes` partitions — fast, approximate, tunable on the accuracy/latency curve. **HNSW** (Hierarchical Navigable Small World) builds a multi-layer proximity graph: the top layer has few nodes and long edges for coarse navigation, each lower layer refines with denser edges, until the bottom layer returns the actual nearest neighbors. Both IVF and HNSW are **approximate** — recall is a knob, not a guarantee — and both ship in Faiss, pgvector, and every dedicated vector database.

---

## Trade-offs and Alternatives

| Decision | Option A | Option B | Prefer A when | Prefer B when |
|---|---|---|---|---|
| OLTP storage engine | B-tree (update-in-place) | LSM-tree (log-structured) | Read-dominated, need predictable latency, want cheap snapshots | Write-heavy, want lower write amplification, can tolerate compaction stalls |
| B-tree crash safety | Write-ahead log + in-place overwrite | Copy-on-write (LMDB) | Standard crash recovery, high concurrency | Want cheap snapshots, willing to trade disk space |
| LSM compaction | Size-tiered | Leveled | Write-dominated, can tolerate 2x transient disk usage | Read-heavy, want smaller read amplification |
| Row storage | Heap file + pointer index | Clustered (row in primary-key tree) | Rows grow in size, many secondary indexes | Read-heavy by primary key, fewer secondary indexes |
| Secondary access path | Normal index | Covering / included-columns index | Writes dominate or extra columns are large | Hot query pattern that reads a small fixed set of columns |
| Analytical storage | Row store | Column store | OLTP queries over current state | Scans over billions of rows, a few columns at a time |
| Column compression | Bitmap encoding | Dictionary / RLE | High cardinality | Low cardinality, long sorted runs |
| Analytical query execution | Interpreted row-by-row | Vectorized or JIT-compiled | Small datasets, tooling simplicity | Millions+ rows per query |
| Precomputation | Query raw data every time | Materialized view / OLAP cube | Ad-hoc, exploratory analytics | Same query runs constantly and is expensive |
| Geospatial index | Concatenated B-tree | R-tree / Bkd-tree / space-filling curve | Only one axis filters | True 2-D bounding-box queries |
| Full-text search | LIKE on a B-tree | Inverted index (Lucene) | Simple prefix search on small data | Real relevance ranking, fuzzy / synonym / phrase queries |
| Vector search | Flat index (exact) | IVF / HNSW (approximate) | Small corpus, perfect recall required | Large corpus, recall-for-latency trade acceptable |

---

## Key Takeaways

1. **OLTP and OLAP storage engines are different species.** Row-oriented B-trees and LSMs serve point queries; columnar compressed storage with vectorized execution serves scans. HTAP systems technically exist but internally maintain both.
2. **LSMs win writes, B-trees win reads — usually.** The reasons are structural: sequential segment writes vs random page writes, lower write amplification vs predictable read latency, background compaction vs fragmentation.
3. **Bloom filters are the small-but-essential optimization** that makes LSM point lookups viable — 10 bits per key for a 1% false-positive rate is the number to remember.
4. **A write-ahead log is the only reason in-place B-trees are crash-safe.** Everything else — page splits, buffering, torn pages — depends on WAL replay to restore a consistent tree.
5. **Indexes are layers on top of a storage engine, not replacements for it.** Secondary / clustered / covering choices trade read hops for write cost and disk space; pick based on the dominant query pattern.
6. **Column stores work because analytical queries read few columns of many rows, and columns compress.** Bitmap encoding plus run-length encoding plus sort order turns terabyte scans into gigabyte scans.
7. **Vectorized and JIT-compiled execution exist because scanning fast is not enough.** Keeping inner loops small, SIMD-friendly, and cache-resident is where OLAP engines actually earn their speed.
8. **Materialized views and OLAP cubes are "do not execute" optimizations** — they pay write-time cost to make repeated read queries nearly free, at the cost of flexibility.
9. **Specialized indexes exist for specialized shapes.** R-trees / Bkd-trees for geo, inverted indexes for text, IVF / HNSW for vector similarity — a 1-D B-tree cannot do any of these.
10. **As an application developer, knowing which family of engine sits underneath** tells you how to tune, which queries will be cheap, and which workloads will be pathological.

---

## See Also

- [[01-log-structured-storage-lsm-trees]]
- [[02-b-trees-and-page-oriented-storage]]
- [[03-comparing-btrees-and-lsm-trees]]
- [[04-secondary-and-clustered-indexes]]
- [[05-column-oriented-storage]]
- [[06-query-execution-and-materialized-views]]
- [[07-multidimensional-and-vector-indexes]]